# Comprehensive Intrinsic Metrics Evaluation
## exp2b_flash_learned_pool — Validation Set Analysis

**Purpose:** Evaluate trained model on validation set without retraining.
Compute micro/macro recall and precision @10/@20, plus breakdowns by code type.

**Model:** `exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt`

### Metrics Computed
1. **Global metrics** — Micro/Macro Recall@10, @20; Micro/Macro Precision@10, @20; NDCG; MRR; Positive Brier
2. **Code-type metrics** — All above broken down by: ICD-10, Procedures, GPI, Provider, Revenue, DRG, Days, Place of Service

### Table of Contents
1. Environment Setup
2. Configuration
3. Load Trained Model
4. Load Validation Data
5. Load w2ind_target & Build Code Type Mapping
6. Comprehensive Evaluation
7. Global Metrics Results
8. Code-Type-Specific Results
9. Visualization


In [ ]:
import sys
import os
import gc
import time
import json
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple, List, Any, Set
from dataclasses import dataclass, field
from collections import defaultdict

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.cloud import bigquery

warnings.filterwarnings("ignore")

MODULE_DIR = os.getcwd()
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

from moe_flashattn_4_core import (
    BaseConfig,
    FlashAttentionConfig,
    MoEConfig,
    ClinicalDatasetLazy,
    create_collate_fn,
    FlashAttentionTransformer,
    FlashMoETransformer,
    BaselineTransformer,
    DataParallelWrapper,
    StreamingMetrics,
    prepare_data_once,
    cleanup_gpu_memory,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {props.total_memory / 1e9:.1f} GB")


## 2. Configuration

### Key Parameters
- Model checkpoint path on GCP Vertex
- Training data table (to reconstruct validation split)
- w2ind_target table (for code type classification)
- K values for top-K metrics
- Train/val split ratio and seed (MUST match training run)


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

TRAINED_MODEL_PATH = (
    "logs/exp_round10_3lobs_formal_training/"
    "exp2b_flash_learned_pool_formal/saved_models/"
    "exp_round10_3lobs_formal_training_exp2b_flash_learned_pool_bs128_ep1_d256_20260312_095916_final.pt"
)

TRAINING_DATA_TABLE = (
    "edp-prod-storage.edp_ent_sdoheir_cns."
    "a834793_Combined_All_LOB_o3_train_ending"
)

W2IND_TARGET_TABLE = (
    "edp-prod-storage.edp_ent_sdoheir_cns."
    "a834793_member_w2ind_target"
)

# CRITICAL: Must match the training run exactly
TRAIN_RATIO = 0.99    # 99% train / 1% validation (formal training)
RANDOM_SEED = 42

# Top-K values for metrics
K_VALUES = (1, 5, 10, 20, 50)
PRIMARY_K_VALUES = (10, 20)  # For detailed reporting

# Evaluation
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 4

# Macro metrics: minimum sample count per code for inclusion
MACRO_MIN_COUNT = 5

print(f"Model: {TRAINED_MODEL_PATH}")
print(f"Training data: {TRAINING_DATA_TABLE}")
print(f"w2ind_target: {W2IND_TARGET_TABLE}")
print(f"Train/Val split: {TRAIN_RATIO}/{1-TRAIN_RATIO:.2f} (seed={RANDOM_SEED})")
print(f"K values: {K_VALUES}")
print(f"Primary K values for detailed report: {PRIMARY_K_VALUES}")
assert os.path.exists(TRAINED_MODEL_PATH), f"Model not found: {TRAINED_MODEL_PATH}"


## 3. Load Trained Model

Load the trained FlashAttentionTransformer from checkpoint.
The checkpoint contains model_state_dict, config, and model_type.


In [ ]:
# ============================================================================
# MODEL LOADING FROM CHECKPOINT
# ============================================================================
# Source: dev/downstream/moe_flashattn_3_lob3_downstream_running.ipynb
#
# Reconstructs the full model architecture from checkpoint metadata and loads
# the saved weights.  Handles all three model families:
#   - BaselineTransformer
#   - FlashAttentionTransformer
#   - FlashMoETransformer (with automatic d_ff inference from expert weights)

def load_model_from_checkpoint(
    model_path: str,
    device: torch.device,
    verbose: bool = True,
) -> Tuple[torch.nn.Module, BaseConfig, Optional[MoEConfig], bool, str]:
    """
    Load a pretrained model from a .pt checkpoint.

    Checkpoint expected keys:
        model_state_dict, model_type, config, moe_config (optional)

    Returns:
        (model, config, moe_config, use_mixed_precision, model_type)
    """
    if verbose:
        print(f"\n{'=' * 70}")
        print(f"Loading model from: {model_path}")

    checkpoint_data = torch.load(model_path, map_location=device, weights_only=False)

    model_type = checkpoint_data.get("model_type", "Unknown")
    config_dict = checkpoint_data.get("config", {})
    moe_config_dict = checkpoint_data.get("moe_config", None)
    state_dict = checkpoint_data["model_state_dict"]

    if verbose:
        print(f"  Model type: {model_type}")
        print(f"  Embedding size: {config_dict.get('embedding_size', 256)}")
        print(f"  N layers: {config_dict.get('nlayers', 6)}")
        print(f"  Learned attention pooling: {config_dict.get('use_learnt_att_pool', False)}")

    # Infer learned pooling from state_dict keys (ground truth)
    use_learnt_att_pool_inferred = "daily_pooling.query" in state_dict

    # For MoE models, infer d_ff from expert weight shapes to avoid mismatch
    inferred_d_ff = None
    if "FlashMoE" in model_type:
        for key in state_dict.keys():
            if "experts.0.ffn.w_gate.weight" in key:
                d_ff_adjusted = state_dict[key].shape[0]
                inferred_d_ff = (d_ff_adjusted * 3 + 1) // 2
                if verbose:
                    print(f"  Inferred d_ff from expert weights: {inferred_d_ff}")
                break
        if inferred_d_ff is None:
            inferred_d_ff = config_dict.get("nhid", 512)

    # For non-MoE FlashAttention models, infer nhid from dense FFN weight shapes
    inferred_nhid = None
    if "FlashMoE" not in model_type:
        for key in state_dict.keys():
            if "temporal_layers.0.ffn.w_gate.weight" in key:
                d_ff_adjusted = state_dict[key].shape[0]
                inferred_nhid = (d_ff_adjusted * 3 + 1) // 2
                if verbose:
                    print(f"  Inferred nhid from FFN weights: {inferred_nhid} "
                          f"(d_ff_adjusted={d_ff_adjusted})")
                break

    # --- Reconstruct model by type ---
    moe_config_out = None

    if "FlashMoE" in model_type:
        config = FlashAttentionConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=config_dict.get("nhid", 512),
            nhead=config_dict.get("nhead", 8),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
            use_learnt_att_pool=use_learnt_att_pool_inferred,
            use_swiglu=config_dict.get("use_swiglu", True),
            use_rope=config_dict.get("use_rope", True),
            use_flash=config_dict.get("use_flash", True),
        )
        d_ff_to_use = inferred_d_ff or config_dict.get("nhid", 512)
        if moe_config_dict:
            if verbose and moe_config_dict.get("d_ff") != d_ff_to_use:
                print(f"  Correcting d_ff: checkpoint={moe_config_dict.get('d_ff')}, actual={d_ff_to_use}")
            moe_config_out = MoEConfig(
                d_model=moe_config_dict.get("d_model", config.embedding_size),
                d_ff=d_ff_to_use,
                num_experts=moe_config_dict.get("num_experts", 8),
                num_shared_experts=moe_config_dict.get("num_shared_experts", 1),
                top_k=moe_config_dict.get("top_k", 2),
                expert_dropout=moe_config_dict.get("expert_dropout", 0.1),
                load_balance_strategy=moe_config_dict.get("load_balance_strategy", "deepseek"),
                aux_loss_weight=moe_config_dict.get("aux_loss_weight", 0.001),
                use_moe_from_layer=moe_config_dict.get("use_moe_from_layer", 2),
                use_swiglu_experts=moe_config_dict.get("use_swiglu_experts", True),
                router_warmup_steps=moe_config_dict.get("router_warmup_steps", 0),
                z_loss_weight=moe_config_dict.get("z_loss_weight", 0.005),
                bias_lr=moe_config_dict.get("bias_lr", 1e-3),
                bias_momentum=moe_config_dict.get("bias_momentum", 0.6),
            )
        else:
            moe_config_out = MoEConfig(d_model=config.embedding_size, d_ff=config.nhid)
        model = FlashMoETransformer(config, moe_config_out)
        use_mixed_precision = True

    elif "FlashAttention" in model_type:
        nhid_to_use = inferred_nhid or config_dict.get("nhid", 512)
        if verbose and config_dict.get("nhid") and inferred_nhid and config_dict["nhid"] != inferred_nhid:
            print(f"  Warning: config nhid={config_dict['nhid']} vs inferred nhid={inferred_nhid}, "
                  f"using weight-inferred value")
        config = FlashAttentionConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=nhid_to_use,
            nhead=config_dict.get("nhead", 8),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
            use_learnt_att_pool=use_learnt_att_pool_inferred,
            use_swiglu=config_dict.get("use_swiglu", True),
            use_rope=config_dict.get("use_rope", True),
            use_flash=config_dict.get("use_flash", True),
        )
        model = FlashAttentionTransformer(config)
        use_mixed_precision = True

    else:
        config = BaseConfig(
            embedding_size=config_dict.get("embedding_size", 256),
            nhid=config_dict.get("nhid", 512),
            nlayers=config_dict.get("nlayers", 6),
            dropout=config_dict.get("dropout", 0.1),
        )
        model = BaselineTransformer(config)
        use_mixed_precision = False

    model.load_state_dict(checkpoint_data["model_state_dict"])
    model = model.to(device)
    model.eval()

    if verbose:
        total_params = sum(p.numel() for p in model.parameters())
        print(f"  Model loaded successfully!")
        print(f"  Total parameters: {total_params:,}")
        print(f"  Mixed precision: {use_mixed_precision}")
        print(f"  Device: {device}")
        print(f"{'=' * 70}\n")

    return model, config, moe_config_out, use_mixed_precision, model_type

In [ ]:
cleanup_gpu_memory(verbose=False)

model, config, moe_config_loaded, use_mixed_precision, model_type = load_model_from_checkpoint(
    model_path=TRAINED_MODEL_PATH,
    device=device,
    verbose=True,
)

print(f"\nConfig summary:")
print(f"  target_cd_cnt: {config.target_cd_cnt}")
print(f"  len_dy: {config.len_dy}")
print(f"  len_cd: {config.len_cd}")
print(f"  cd_cnt: {config.cd_cnt}")
print(f"  embedding_size: {config.embedding_size}")
